# Domain-adapting MobileCLIP2-S2 on Pl@ntNet-300K

**Runtime → Change runtime type → A100.**

First run ~2.5–3.5 h (fetch 31.7 GB, resize, train). **Every run after that skips
the download entirely** — the raw archive, a decode-fast copy of the dataset, and
the training checkpoints all live on your Drive, so a disconnect costs minutes.

## What this is testing

The 17.9 MB encoder scores **0.6236** species top-1 on the 490-class catalogue;
`plantclef24` at 43 MB scores **0.7671**. Three inference-side levers were measured
and all three are null (`SMALL_FRONTIER_FINDINGS.md`), so that 14.4pp sits in the
encoder's representation.

What `plantclef24` *is*, is a DINOv2 ViT-B fine-tuned on 7,806 Pl@ntNet species —
stock DINOv2 ViT-B is unremarkable here. **So the 43 MB advantage is domain
adaptation, not size**, and nobody has done that to a 17.9 MB model. This runs the
same recipe one encoder-scale down.

Predictions are in `ADAPT_PREREG.md`, written before any run: **0.65–0.72** species
against stock 0.6236, genus moving more than species, **0.74+ changes the size
decision**.

> **The trap this is built to avoid.** Training on the catalogue's own species and
> then reporting catalogue accuracy cannot tell *learned plants* from *learned this
> label set*. So 551 Pl@ntNet species are kept out of training entirely, and a
> linear probe on them runs every epoch beside the loss. A species gain with
> transfer *below stock* means a catalogue-specific encoder — a much weaker claim.


## 1. Environment and Drive

Everything durable goes in one Drive folder: the compact dataset, the training
checkpoints, and the final tower.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install open_clip_torch==2.32.0 pandas pyarrow


In [ ]:
import os, json, time, tarfile, zipfile, pathlib, subprocess, shutil
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from google.colab import drive
drive.mount('/content/drive')

DRIVE = pathlib.Path('/content/drive/MyDrive/plantnet_adapt')
DRIVE.mkdir(parents=True, exist_ok=True)
DEV = 'cuda'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(torch.__version__, torch.cuda.get_device_name(0))
print('durable dir:', DRIVE)
free = shutil.disk_usage('/content').free / 1e9
print(f'local scratch free: {free:.0f} GB')


## 2. The research repo


In [ ]:
REPO = pathlib.Path('narrowcast-plantid')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/semajyllek/narrowcast-plantid.git'], check=True)
print(open(REPO / 'ADAPT_PREREG.md').read()[:900])


## 3. Dataset — two levels of Drive cache

Both live on Drive, so **the Zenodo download happens exactly once, ever**:

| on Drive | size | why |
|---|---|---|
| `plantnet_300K.zip` | 31.7 GB | the archive, so Zenodo is never hit twice |
| `plantnet_288.tar` | ~10 GB | every image resized; what training actually reads |

The resized copy is **not** a space compromise — it is a throughput one. A 17.9 MB
model on an A100 is starved by JPEG decode long before it is starved by compute,
and full-resolution Pl@ntNet images make the dataloader the bottleneck. 288 px on
the short side against a 256 px training crop loses nothing the model can see, and
roughly triples steps per second.

A session restores from the tar in a few minutes. If the resize ever needs redoing,
the raw zip is already local to Drive and Zenodo is not involved.


In [ ]:
RESIZE = 288
RAW = DRIVE / 'plantnet_300K.zip'          # 31.7 GB, fetched once
TAR = DRIVE / f'plantnet_{RESIZE}.tar'     # ~10 GB, what training reads
SMALL = pathlib.Path('/content/plantnet_small')
for f in (RAW, TAR):
    print(f'{f.name:24s} {"present" if f.exists() else "missing"}')


In [ ]:
def fetch_raw():
    """31.7 GB from Zenodo, straight onto Drive. Resumable, and runs once ever."""
    URL = 'https://zenodo.org/api/records/5645731/files/plantnet_300K.zip/content'
    if RAW.exists() and RAW.stat().st_size > 31e9:
        print(f'raw archive already on Drive ({RAW.stat().st_size/1e9:.1f} GB)')
        return
    print('downloading 31.7 GB to Drive (resumable -- just rerun if it drops)...')
    subprocess.run(['curl', '-L', '-C', '-', '--retry', '5', '--retry-delay', '5',
                    '-o', str(RAW), URL], check=True)
    print(f'{RAW.stat().st_size/1e9:.1f} GB on Drive')


def build_cache():
    """Extract the raw archive and write the resized tar to Drive. Runs once."""
    fetch_raw()
    raw = pathlib.Path('/content/plantnet_raw')
    if not raw.exists():
        t0 = time.time()
        with zipfile.ZipFile(RAW) as z:
            z.extractall(raw)
        print(f'extracted in {time.time()-t0:.0f}s')

    images = next(p for p in raw.rglob('images') if p.is_dir())
    files = sorted(images.rglob('*.jpg'))
    # Everything is cached: the per-species cap is a training knob applied later,
    # so it can be changed without rebuilding this.
    print(f'{len(files):,} images to resize')

    from multiprocessing import Pool
    SMALL.mkdir(parents=True, exist_ok=True)
    jobs = [(str(p), str(SMALL / p.parent.parent.name / p.parent.name / p.name))
            for p in files]
    for d in {pathlib.Path(j[1]).parent for j in jobs}:
        d.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with Pool(os.cpu_count()) as pool:
        for i, _ in enumerate(pool.imap_unordered(_shrink, jobs, chunksize=256)):
            if i % 25000 == 0:
                print(f'  resized {i:,}/{len(jobs):,} ({time.time()-t0:.0f}s)', flush=True)
    print(f'resized in {time.time()-t0:.0f}s')

    t0 = time.time()
    tmp = pathlib.Path('/content/cache.tar')
    with tarfile.open(tmp, 'w') as t:
        t.add(SMALL, arcname='.')
    print(f'tar {tmp.stat().st_size/1e9:.1f} GB in {time.time()-t0:.0f}s; copying to Drive...')
    shutil.copy(tmp, TAR)
    print('cached at', TAR)


In [ ]:
# module-level so multiprocessing can pickle it
def _shrink(job):
    src, dst = job
    try:
        im = Image.open(src).convert('RGB')
        w, h = im.size
        s = RESIZE / min(w, h)
        if s < 1:
            im = im.resize((round(w * s), round(h * s)), Image.BICUBIC)
        im.save(dst, 'JPEG', quality=88)
    except Exception:
        pass


In [ ]:
# TAR -> restore in minutes.  RAW on Drive -> resize only.  Neither -> fetch once.
if TAR.exists():
    if not SMALL.exists():
        t0 = time.time()
        SMALL.mkdir(parents=True, exist_ok=True)
        with tarfile.open(TAR) as t:
            t.extractall(SMALL)
        print(f'restored from Drive in {time.time()-t0:.0f}s')
    else:
        print('already unpacked locally')
else:
    build_cache()

n = sum(1 for _ in SMALL.rglob('*.jpg'))
print(f'{n:,} images ready at {SMALL}')


## 4. The training set

Labels come from the directory structure — `<split>/<species_id>/<id>.jpg` — rather
than the metadata json, which the archive does not always carry. Pl@ntNet's own
split is kept: `val` watches for overfitting and nothing is selected on `test`.

**Training is restricted to the 530 catalogue species, and this is load-bearing.**
The other 551 species are reserved so that §7's probe measures transfer to species
the tower has *never seen*. Fine-tuning on all 1,081 and then probing 551 of them
would measure nothing — they would be in-training-distribution, and the probe would
pass by construction.

It costs almost no data: the catalogue was selected by image availability and holds
299,832 of the 306,146 images. The 551 reserved species carry about nine each.


In [ ]:
MAX_PER_SPECIES = 200   # training-set balance knob; None uses every image

rows = [(p.stem, p.parent.name, p.parent.parent.name, str(p))
        for p in SMALL.rglob('*.jpg')]
everything = pd.DataFrame(rows, columns=['image_id', 'species_id', 'split', 'path'])
assert set(everything['split']) <= {'train', 'val', 'test'}

cat_ids = set(json.load(open(REPO / 'catalogue_species.json'))['species'])
df = everything[everything.species_id.isin(cat_ids)].copy()      # trained on
unseen = everything[~everything.species_id.isin(cat_ids)].copy()  # never trained on
print(f'train pool : {df.species_id.nunique()} catalogue species, {len(df):,} images')
print(f'reserved   : {unseen.species_id.nunique()} species, {len(unseen):,} images'
      ' (never seen -- the probe set)')

classes = sorted(df['species_id'].unique())
cls_idx = {c: i for i, c in enumerate(classes)}
df['y'] = df['species_id'].map(cls_idx)
unseen['y'] = 0   # unused; the probe fits its own head

# Applied to `train` only, and only here -- the cache holds everything, so this
# can change without touching Drive. Pl@ntNet-300K is severely imbalanced and an
# uncapped objective is dominated by a handful of species.
if MAX_PER_SPECIES:
    # shuffle-then-head rather than groupby.apply(sample): apply() consumes the
    # key column in pandas 3 and this has to work on whatever Colab ships.
    tr_rows = (df[df.split == 'train'].sample(frac=1, random_state=0)
                 .groupby('species_id', sort=False).head(MAX_PER_SPECIES))
    df = pd.concat([tr_rows, df[df.split != 'train']], ignore_index=True)

print(f'{len(df):,} images, {len(classes)} species')
print(df['split'].value_counts().to_dict())
n_tr = df[df.split=='train'].groupby('species_id').size()
print(f'train images/species: min {n_tr.min()} median {int(n_tr.median())} max {n_tr.max()}')


## 5. Model

Only `.visual` is trained; a linear classifier is bolted on for the objective and
discarded afterwards. The text tower is dropped — it never ships.


In [ ]:
import open_clip
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    'hf-hub:timm/MobileCLIP2-S2-OpenCLIP')
tower = model.visual.to(DEV)
del model
with torch.no_grad():
    d = tower(torch.zeros(1, 3, 256, 256, device=DEV)).shape[-1]
print('embedding dim', d,
      f'| params {sum(p.numel() for p in tower.parameters())/1e6:.1f}M')
head = nn.Linear(d, len(classes)).to(DEV)


## 6. Data pipeline


In [ ]:
class PN(Dataset):
    def __init__(self, frame, tf):
        self.p = frame['path'].tolist(); self.y = frame['y'].tolist(); self.tf = tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        try:
            im = Image.open(self.p[i]).convert('RGB')
        except Exception:
            im = Image.new('RGB', (256, 256))
        return self.tf(im), self.y[i]

tr = DataLoader(PN(df[df.split=='train'], preprocess_train), batch_size=256,
                shuffle=True, num_workers=8, pin_memory=True, drop_last=True,
                persistent_workers=True)
va = DataLoader(PN(df[df.split=='val'], preprocess_val), batch_size=512,
                shuffle=False, num_workers=8, pin_memory=True)
print(len(tr), 'train batches |', len(va), 'val batches')


## 7. The metric that actually matters, set up before training

Val accuracy is a **training diagnostic and nothing more**: 1,081-way, per image,
on Pl@ntNet, using the classifier that gets thrown away. It is not comparable to
anything in `CLAUDE.md`, and it is flattering — val is severely imbalanced, with
the top 50 species holding 57% of its 31,118 images.

Worse, **a rising val number does not mean the frozen tower got more useful**.
Cross-entropy fine-tuning routinely sharpens features for the training label set
while making them *less* linearly transferable to a different one — which is
exactly the failure that would make this experiment worthless. The standard
evaluation cannot see it, because 530 of the 1,081 training species *are* the
catalogue.

So the unseen-species probe runs **every epoch**, next to val. If transfer
degrades at epoch 2 you will see it at epoch 2, not after the full run.

It is a real test only because §4 kept those 551 species out of training. Probing
species the tower was fine-tuned on would pass by construction and mean nothing.


In [ ]:
# `unseen` never enters training -- see section 4. That is what makes this a test.
probe_set = unseen.groupby('species_id').filter(lambda g: len(g) >= 6)
probe_y = probe_set['species_id'].astype('category').cat.codes.to_numpy()
print(f'probe: {probe_set.species_id.nunique()} unseen species, {len(probe_set):,} images')
assert not set(probe_set.species_id) & set(classes), 'probe species leaked into training'


In [ ]:
def embed_all(tw, frame):
    dl = DataLoader(PN(frame, preprocess_val), batch_size=512, num_workers=8)
    was_training = tw.training
    tw.eval(); out = []
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        for x, _ in dl:
            out.append(F.normalize(tw(x.to(DEV)).float(), dim=-1).cpu().numpy())
    if was_training:
        tw.train()
    return np.vstack(out)


def probe(X, y, seed=0):
    """Linear probe on held-out species: can a fresh head still read this tower?"""
    from sklearn.linear_model import LogisticRegression
    rng = np.random.default_rng(seed); idx = rng.permutation(len(X))
    cut = int(0.6 * len(X)); a, b = idx[:cut], idx[cut:]
    clf = LogisticRegression(max_iter=2000, C=10.0).fit(X[a], y[a])
    return float((clf.predict(X[b]) == y[b]).mean())


def transfer(tw):
    return probe(embed_all(tw, probe_set), probe_y)


In [ ]:
# The stock baseline, computed once. Every later number is read against it.
STOCK_PROBE = DRIVE / 'stock_probe.json'
if STOCK_PROBE.exists():
    stock_transfer = json.load(open(STOCK_PROBE))['stock']
else:
    _stock, _, _ = open_clip.create_model_and_transforms(
        'hf-hub:timm/MobileCLIP2-S2-OpenCLIP')
    stock_transfer = transfer(_stock.visual.to(DEV))
    del _stock
    torch.cuda.empty_cache()
    json.dump({'stock': stock_transfer}, open(STOCK_PROBE, 'w'))
print(f'stock tower, held-out species probe: {stock_transfer:.4f}')
print('Adaptation must beat this. If it does not, the tower learned the'
      ' catalogue rather than plants.')


## 8. Train, watching transfer as well as val

Cross-entropy, AdamW, one-cycle, bf16. The tower gets a lower learning rate than
the fresh head — a randomly initialised classifier would otherwise wreck the
pretrained features in the first few hundred steps.

Each epoch prints **val** (the diagnostic) and **transfer** (the thing that
decides whether this worked), and checkpoints both to Drive.

**If the runtime dies, re-run this cell** — it reloads the last epoch and carries
on. **If transfer falls for two epochs running, stop**: the tower is specialising
onto the training label set and more epochs will not fix it.


In [ ]:
EPOCHS = 4
CKPT = DRIVE / 'checkpoint.pt'

opt = torch.optim.AdamW([{'params': tower.parameters(), 'lr': 1e-4},
                         {'params': head.parameters(),  'lr': 1e-3}],
                        weight_decay=0.05)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=[1e-4, 1e-3], total_steps=EPOCHS * len(tr), pct_start=0.1)

history, start_ep = [], 0
if CKPT.exists():
    st = torch.load(CKPT, map_location=DEV)
    tower.load_state_dict(st['tower']); head.load_state_dict(st['head'])
    opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
    history, start_ep = st.get('history', []), st['epoch'] + 1
    print(f"resumed at epoch {start_ep} (val {st['val']:.4f}, "
          f"transfer {st.get('transfer', float('nan')):.4f})")


def val_acc():
    tower.eval(); head.eval(); ok = n = 0
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        for x, y in va:
            ok += (head(tower(x.to(DEV))).argmax(1).cpu() == y).sum().item(); n += len(y)
    tower.train(); head.train()
    return ok / max(n, 1)


t0 = time.time()
for ep in range(start_ep, EPOCHS):
    tower.train(); head.train()
    for i, (x, y) in enumerate(tr):
        x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss = F.cross_entropy(head(tower(x)), y, label_smoothing=0.1)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if i % 100 == 0:
            print(f'  ep{ep} {i}/{len(tr)} loss {loss.item():.3f} ({time.time()-t0:.0f}s)', flush=True)

    v, tx = val_acc(), transfer(tower)
    history.append({'epoch': ep, 'val': v, 'transfer': tx})
    torch.save({'tower': tower.state_dict(), 'head': head.state_dict(),
                'opt': opt.state_dict(), 'sched': sched.state_dict(),
                'epoch': ep, 'val': v, 'transfer': tx, 'history': history}, CKPT)
    flag = '' if tx >= stock_transfer else '   <-- BELOW STOCK'
    print(f'epoch {ep}: val {v:.4f} | transfer {tx:.4f} (stock {stock_transfer:.4f}, {tx-stock_transfer:+.4f}){flag}', flush=True)


In [ ]:
pd.DataFrame(history).assign(stock=stock_transfer,
                            delta=lambda d: d.transfer - stock_transfer)


## 9. Save the tower

The classifier is discarded: the product is a frozen feature extractor, and keeping
a 1,081-way head would only invite someone to use it.

**Save the epoch with the best transfer, not the last one** — if transfer peaked at
epoch 2 the later weights are worse for the only purpose this tower has.


In [ ]:
best = max(history, key=lambda h: h['transfer']) if history else None
if best and best['epoch'] != history[-1]['epoch']:
    print(f"NOTE: transfer peaked at epoch {best['epoch']} "
          f"({best['transfer']:.4f}) but the loaded weights are epoch "
          f"{history[-1]['epoch']}. Re-run training with EPOCHS={best['epoch']+1} "
          'after deleting the checkpoint, or accept the later weights knowingly.')

OUT = DRIVE / 'mobileclip2_s2_plantnet.pt'
torch.save({k: v.cpu() for k, v in tower.state_dict().items()}, OUT)
print(f'{OUT} — {OUT.stat().st_size/1e6:.1f} MB (on Drive, survives disconnect)')
try:
    from google.colab import files
    files.download(str(OUT))
except Exception as e:
    print('grab it from Drive instead:', e)


## 10. Back on your machine

```bash
mkdir -p data/processed/adapted
cp ~/Downloads/mobileclip2_s2_plantnet.pt data/processed/adapted/

PYTHONPATH=. .venv-mps/bin/python -c "
from plantid.features import embed_catalog, embed_inat, embed_background
for f in (embed_catalog.main, embed_inat.main, embed_background.main):
    f('mobileclip2_s2_ft')"

PYTHONPATH=. .venv/bin/python -m plantid.eval.rejection --variant mobileclip2_s2_ft
```

That prints observation-level species and genus top-1 over the 490-class catalogue
on 3,435 held-out iNaturalist observations — directly comparable to:

| encoder | MB | species | genus |
|---|---|---|---|
| `mobileclip2_s2` (stock) | 17.9 | 0.6236 | 0.8122 |
| `plantclef24` | 43.3 | 0.7671 | 0.9258 |
| `bioclip2_cml4` | 160 | 0.8370 | 0.9735 |

and to the predictions in `ADAPT_PREREG.md`: **0.65–0.72**, genus moving more than
species, **0.74+ changes the size decision**.

Report the transfer probe alongside it. A species gain with transfer *below* stock
means a catalogue-specific encoder, which is a different and much weaker claim.
